# Tablero de completitud de cuentas contables — DWH

**¿Qué hace este notebook?** Se conecta al Data Warehouse (DWH) y revisa, para el **último año
(12 meses)**, si el campo `VALOR_RESERVA_CONTABLE` (la plata que la aseguradora aparta como
respaldo) tiene valor o está vacío, por cada combinación de **fuente + cuenta contable + libro**.

Es la versión con historia del control `control_completitud_cuentas.sql` (que revisa un solo mes).

**¿Cuándo correrlo?** Los días **8 o 9** de cada mes, antes del proceso oficial de cierre (día 10),
para poder avisar a tiempo si a alguna cuenta le falta información.

**Cómo leer el resultado (semáforo):**
- 🟢 **COMPLETO** = todos los registros de esa cuenta/libro/mes tienen valor → OK para cerrar.
- 🔴 **ALERTA** = hay registros con el valor vacío (nulo) → avisar al dueño de la tabla.

**Cuentas que NO se revisan aquí** (tienen sus propios controles): Incurrido, Salvamentos,
Recobros, IVA AG (se excluye con el filtro `Libro <> 'AG'`) + 2 cuentas por confirmar con Andrey.

> Las credenciales se leen de `credenciales_local.py`, que está en el `.gitignore`
> y **nunca** debe subirse al repositorio.

In [11]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Credenciales locales (archivo ignorado por git, vive solo en este computador)
from credenciales_local import (
    DEFAULT_HOST_DWH, DEFAULT_PORT_DWH, DEFAULT_DATABASE_DWH,
    DEFAULT_USER_DWH, DEFAULT_PASSWORD_DWH,
)

## 1. Conexión al DWH

Creamos la conexión al servidor SQL Server del Data Warehouse y hacemos una prueba mínima
(`SELECT 1`): si la celda imprime `conexion_ok = 1`, la conexión funciona.

In [12]:
url = URL.create(
    "mssql+pyodbc",
    username=DEFAULT_USER_DWH,
    password=DEFAULT_PASSWORD_DWH,
    host=DEFAULT_HOST_DWH,
    port=int(DEFAULT_PORT_DWH),
    database=DEFAULT_DATABASE_DWH,
    query={"driver": "SQL Server"},  # driver ODBC disponible en esta máquina
)
engine = create_engine(url)

pd.read_sql("SELECT 1 AS conexion_ok", engine)

,conexion_ok
0,1


## 2. ¿Qué meses vamos a revisar?

En vez de escribir el periodo a mano, le preguntamos a las propias tablas cuál es el **último
periodo contable cargado** (formato `AAAAMM`, por ejemplo `202607` = julio de 2026) y desde ahí
contamos **12 meses hacia atrás**. Así el tablero siempre muestra el último año disponible sin
tener que editar nada.

> Solo se consulta el máximo en `DIRECTA` y `CEDIDAS_IAXIS`, que responden en segundos.
> Las otras dos tablas (`CEDIDAS_AS400` y `CEDIDAS_TERREMOTO`) tardan varios minutos incluso
> para un simple `MAX` (no tienen índice por periodo), y comparten el mismo calendario de carga.

In [13]:
sql_ultimo_periodo = """
SELECT MAX(p) AS ultimo_periodo
FROM (
    SELECT MAX(periodo_contable_analisis) AS p FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ
    UNION ALL
    SELECT MAX(periodo_contable_analisis)      FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS
) t
"""
ULTIMO_PERIODO = int(pd.read_sql(sql_ultimo_periodo, engine).iloc[0, 0])


def periodos_hacia_atras(periodo_final: int, n: int = 12) -> list[int]:
    """Lista de n periodos AAAAMM terminando en periodo_final (incluido)."""
    anio, mes = divmod(periodo_final, 100)
    periodos = []
    for _ in range(n):
        periodos.append(anio * 100 + mes)
        mes -= 1
        if mes == 0:
            anio, mes = anio - 1, 12
    return sorted(periodos)


PERIODOS = periodos_hacia_atras(ULTIMO_PERIODO, 12)
print(f"Último periodo cargado en DWH: {ULTIMO_PERIODO}")
print(f"Ventana de análisis (12 meses): {PERIODOS[0]} → {PERIODOS[-1]}")

Último periodo cargado en DWH: 202606
Ventana de análisis (12 meses): 202507 → 202606


## 3. El control de completitud (12 meses en una sola consulta)

Es exactamente la misma lógica de `control_completitud_cuentas.sql` — mismas 4 fuentes, mismas
cuentas, mismos filtros (`Libro <> 'AG'`, etc.) — con un solo cambio: en vez de revisar **un**
periodo, revisa los **12** de la ventana.

Columnas del resultado:
- `total_registros`: cuántas filas hay para esa cuenta/libro/mes.
- `con_valor` / `sin_valor`: cuántas tienen el valor de la reserva y cuántas lo tienen vacío (nulo).
- `pct_completitud`: porcentaje con valor (100 = perfecto).
- `semaforo`: COMPLETO o ALERTA.

> ⏳ **Paciencia**: esta celda puede tardar **varios minutos** — dos de las tablas no tienen
> índice por periodo y el motor tiene que recorrerlas completas. Es normal.

In [14]:
# Los periodos son enteros calculados por nosotros (no texto del usuario),
# por eso es seguro insertarlos directamente en el SQL.
lista_periodos = ", ".join(str(int(p)) for p in PERIODOS)

query_control = f"""
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    total_registros,
    con_valor,
    sin_valor,
    CAST(ROUND(100.0 * con_valor / NULLIF(total_registros, 0), 2) AS DECIMAL(5,2)) AS pct_completitud,
    semaforo
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        COUNT(*)        AS total_registros,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END) AS con_valor,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END) AS sin_valor,
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END AS semaforo
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400 */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
"""

df = pd.read_sql(query_control, engine)
df["pct_completitud"] = df["pct_completitud"].astype(float)
df["pct_sin_valor"] = (100.0 * df["sin_valor"] / df["total_registros"]).round(2)

print(f"{len(df)} filas (combinaciones fuente + cuenta + libro + mes)")
df.head(10)

240 filas (combinaciones fuente + cuenta + libro + mes)


,fuente,cuenta,libro,periodo_contable_analisis,total_registros,con_valor,sin_valor,pct_completitud,semaforo,pct_sin_valor
0,DIRECTA_IAXIS,410310,AL,202603,1513565,1513565,0,100.0,COMPLETO,0.0
1,DIRECTA_IAXIS,410315,AL,202604,81474,81474,0,100.0,COMPLETO,0.0
2,DIRECTA_IAXIS,510305,AL,202603,8105886,8105886,0,100.0,COMPLETO,0.0
3,DIRECTA_IAXIS,510305,AL,202604,9259457,9259457,0,100.0,COMPLETO,0.0
4,DIRECTA_IAXIS,510310,AL,202604,2143781,2143781,0,100.0,COMPLETO,0.0
5,DIRECTA_IAXIS,510315,AL,202507,59243,59243,0,100.0,COMPLETO,0.0
6,DIRECTA_IAXIS,410305,AL,202508,3243169,3243169,0,100.0,COMPLETO,0.0
7,DIRECTA_IAXIS,410305,AL,202603,8976620,8976620,0,100.0,COMPLETO,0.0
8,DIRECTA_IAXIS,410305,AL,202605,7286000,7286000,0,100.0,COMPLETO,0.0
9,DIRECTA_IAXIS,510310,AL,202603,1919823,1919823,0,100.0,COMPLETO,0.0


## 4. Reglas de calidad adicionales — variación y volumen

Hasta acá el tablero solo revisaba **si el dato existe** (completitud). Ahora agregamos una revisión
distinta: **¿se movió más de lo normal?** — esto es justo lo que en `Hallazgos y Estado del Proyecto.md`
se llama la "Fase 2" (detectar valores atípicos), que todavía no estaba definida. Lo que sigue es una
primera propuesta simple; valídala con el equipo antes de confiar en ella del todo.

Vamos a revisar tres cosas nuevas:

- **4.1 Variación de valor**: ¿la plata total de la reserva contable de una cuenta/libro cambió mucho
  de un mes a otro?
- **4.2 Variación de volumen**: ¿la cantidad de registros (filas) cambió mucho? A veces el problema no
  es el valor sino que "desaparecieron" o "aparecieron" registros de la nada.
- **4.3 Huecos de carga**: ¿hubo algún mes en que una cuenta/libro no tuvo **ni una sola fila**? Esto
  casi siempre significa que esa carga no corrió ese mes, más que un problema del dato en sí.

Una combinación se marca como **"variación atípica"** si pasa cualquiera de estas dos pruebas:

1. Se aleja más de **2 desviaciones estándar** de su propio promedio en la ventana de 12 meses (el
   "z-score" — no te preocupes por la fórmula exacta: el número solo dice qué tan raro es este valor
   comparado con lo que esa misma cuenta suele tener).
2. Cambió más de **20%** respecto al mes inmediatamente anterior.

Ambos umbrales están definidos como variables (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION`) al inicio
del siguiente bloque de código — súbelos si el tablero avisa demasiado, o bájalos si avisa muy poco.

In [15]:
df["serie"] = df["fuente"] + " · " + df["cuenta"].astype(str) + " · " + df["libro"].astype(str).str.strip()

# UMBRALES: primera propuesta, hay que validarlos con el equipo — esto es la "Fase 2"
# que menciona Hallazgos y Estado del Proyecto.md y que todavía no estaba definida.
UMBRAL_Z_VARIACION = 2.0     # desviaciones estándar respecto al promedio de la propia serie
UMBRAL_PCT_VARIACION = 0.20  # 20% de cambio respecto al mes inmediatamente anterior


def marcar_variacion_atipica(pivot: pd.DataFrame, periodos: list[int],
                              umbral_z: float = UMBRAL_Z_VARIACION,
                              umbral_pct: float = UMBRAL_PCT_VARIACION,
                              nombre_id: str = "serie") -> pd.DataFrame:
    """
    Recibe una tabla (serie x periodo) y devuelve, en formato largo, una fila por
    serie+periodo con: el valor, su z-score respecto al promedio de esa misma serie
    en la ventana, la variación % contra el mes anterior, y si se marca atípica.
    """
    promedio = pivot.mean(axis=1)
    desviacion = pivot.std(axis=1)
    filas = []
    for idx in pivot.index:
        valores = pivot.loc[idx]
        for i, periodo in enumerate(periodos):
            valor = valores.get(periodo)
            if pd.isna(valor):
                continue
            d = desviacion[idx]
            z = (valor - promedio[idx]) / d if d and not pd.isna(d) else float("nan")
            anterior = valores.get(periodos[i - 1]) if i > 0 else None
            if anterior is None or pd.isna(anterior):
                pct = float("nan")
            elif anterior == 0:
                pct = float("inf") if valor != 0 else 0.0
            else:
                pct = (valor - anterior) / abs(anterior)
            atipica = (not pd.isna(z) and abs(z) > umbral_z) or (not pd.isna(pct) and abs(pct) > umbral_pct)
            filas.append({
                nombre_id: idx,
                "periodo_contable_analisis": periodo,
                "valor": valor,
                "z_score": z,
                "pct_variacion": pct,
                "variacion_atipica": bool(atipica),
            })
    return pd.DataFrame(filas)

### 4.1 Variación de valor

Necesitamos una consulta nueva: la de la sección 3 solo cuenta cuántos registros tienen valor o no
(para completitud), pero no suma el valor en sí. Aquí sumamos `VALOR_RESERVA_CONTABLE` por fuente +
cuenta + libro + mes — misma estructura de 4 fuentes y mismos filtros que la sección 3, solo que
sumando en vez de contando nulos.

In [16]:
query_variacion = f"""
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    valor_total
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6))) AS valor_total
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400 */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
"""

df_valor = pd.read_sql(query_variacion, engine)
df_valor["serie"] = (df_valor["fuente"] + " · " + df_valor["cuenta"].astype(str)
                     + " · " + df_valor["libro"].astype(str).str.strip())

piv_valor = (df_valor.pivot_table(index="serie", columns="periodo_contable_analisis",
                                  values="valor_total", aggfunc="sum")
             .reindex(columns=PERIODOS))

variacion_valor = marcar_variacion_atipica(piv_valor, PERIODOS)
alertas_valor = (variacion_valor[variacion_valor["variacion_atipica"]]
                 .sort_values("periodo_contable_analisis", ascending=False)
                 .reset_index(drop=True))

print(f"{len(df_valor)} filas de valor total (fuente + cuenta + libro + mes)")
print(f"{len(alertas_valor)} variaciones atípicas de VALOR en la ventana de 12 meses")
alertas_valor.head(10)

240 filas de valor total (fuente + cuenta + libro + mes)
153 variaciones atípicas de VALOR en la ventana de 12 meses


,serie,periodo_contable_analisis,valor,z_score,pct_variacion,variacion_atipica
0,CEDIDAS_AS400 · 510305 · AA,202606,8.083954e+08,1.409595,3.575248,True
1,CEDIDAS_IAXIS · 510305 · AL,202606,4.441608e+08,0.373344,-0.666169,True
2,DIRECTA_IAXIS · 510310 · AA,202606,2.259010e+09,1.365615,0.593122,True
3,CEDIDAS_TERREMOTO · 510305 · AL,202606,-5.958562e+08,0.191155,0.868762,True
4,CEDIDAS_TERREMOTO · 510305 · AA,202606,-4.896680e+08,-1.120584,-3.142477,True
5,CEDIDAS_TERREMOTO · 410305 · AL,202606,1.135412e+09,0.244918,-0.290825,True
6,CEDIDAS_TERREMOTO · 410305 · AA,202606,-5.096016e+06,0.037570,-1.002011,True
7,CEDIDAS_TERREMOTO · 263005 · AL,202606,-5.395555e+08,-0.507411,-1.183569,True
8,CEDIDAS_TERREMOTO · 263005 · AA,202606,4.947640e+08,0.286739,1.179068,True
9,DIRECTA_IAXIS · 410310 · AA,202606,-7.624850e+08,0.182717,0.231039,True


### 4.2 Variación de volumen

No necesitamos otra consulta: la columna `total_registros` ya viene en el `df` de la sección 3. Aquí
solo aplicamos la misma prueba de variación atípica, pero sobre la cantidad de filas en vez del valor
monetario.

In [17]:
piv_vol = (df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
           .reindex(columns=PERIODOS))

variacion_vol = marcar_variacion_atipica(piv_vol, PERIODOS)
alertas_volumen = (variacion_vol[variacion_vol["variacion_atipica"]]
                   .sort_values("periodo_contable_analisis", ascending=False)
                   .reset_index(drop=True))

print(f"{len(alertas_volumen)} variaciones atípicas de VOLUMEN (cantidad de registros) en la ventana de 12 meses")
alertas_volumen.head(10)

83 variaciones atípicas de VOLUMEN (cantidad de registros) en la ventana de 12 meses


,serie,periodo_contable_analisis,valor,z_score,pct_variacion,variacion_atipica
0,CEDIDAS_TERREMOTO · 263005 · AL,202606,90568,-0.484468,-0.258709,True
1,CEDIDAS_TERREMOTO · 263005 · AA,202606,32,-0.688636,-0.959698,True
2,CEDIDAS_TERREMOTO · 410305 · AA,202606,1,-0.651650,-0.998561,True
3,CEDIDAS_TERREMOTO · 510305 · AL,202606,26503,-0.190418,0.210293,True
4,CEDIDAS_TERREMOTO · 510305 · AA,202606,31,-0.556228,-0.686869,True
5,CEDIDAS_TERREMOTO · 410305 · AL,202606,64065,-0.351629,-0.361126,True
6,DIRECTA_IAXIS · 410310 · AA,202606,9570,-0.229576,-0.250587,True
7,CEDIDAS_TERREMOTO · 510305 · AL,202605,21898,-0.299016,0.871304,True
8,DIRECTA_IAXIS · 510315 · AL,202605,29679,-0.619880,0.213071,True
9,CEDIDAS_TERREMOTO · 510305 · AA,202604,90,-0.553861,1.432432,True


### 4.3 Huecos de carga

Un "hueco" es distinto de una "alerta de completitud": una alerta significa que llegaron filas pero
algunas tienen el valor vacío; un hueco significa que **no llegó ninguna fila** para esa cuenta/libro
en ese mes — probablemente la carga de esa fuente no corrió.

In [18]:
piv_tot = (df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
           .reindex(columns=PERIODOS))

huecos = (piv_tot.reset_index()
          .melt(id_vars="serie", var_name="periodo_contable_analisis", value_name="total_registros"))
huecos = (huecos[huecos["total_registros"].isna()]
          .drop(columns="total_registros")
          .sort_values(["serie", "periodo_contable_analisis"])
          .reset_index(drop=True))

print(f"{len(huecos)} huecos de carga (combinación fuente+cuenta+libro sin ninguna fila en ese mes)")
huecos.head(10)

0 huecos de carga (combinación fuente+cuenta+libro sin ninguna fila en ese mes)


,serie,periodo_contable_analisis


## 5. Tablero HTML — un solo archivo para compartir

Todo lo que sigue arma un **archivo `.html` autocontenido** con los KPIs, el mapa de calor, las
tendencias, la variación con anomalías marcadas, y las tablas de alertas — para poder abrirlo con
doble clic en cualquier navegador o mandarlo por correo, sin que la otra persona necesite Python ni
acceso al DWH.

> **Nota de diseño**: por simplicidad, los gráficos quedan fijos en modo claro (la misma paleta
> validada que ya usaba este notebook). Lo que sí se adapta a modo oscuro automáticamente es el resto
> de la página (fondo, texto, tarjetas de KPIs, tablas) — si tu correo o navegador usa tema oscuro, el
> tablero no se va a ver "roto", solo los gráficos van a mantener su fondo claro.

In [19]:
# --- paleta (validada; ver skill dataviz / references/palette.md) ---
SURFACE  = "#fcfcfb"   # fondo de los gráficos
INK      = "#0b0b0b"   # texto principal
INK_2    = "#52514e"   # texto secundario
MUTED    = "#898781"   # ejes y etiquetas menores
GRID     = "#e1e0d9"   # líneas de la cuadrícula
FONT     = 'system-ui, -apple-system, "Segoe UI", sans-serif'

# Un color fijo por fuente (nunca cambia aunque se filtre)
COLOR_FUENTE = {
    "DIRECTA_IAXIS":     "#2a78d6",  # azul
    "CEDIDAS_AS400":     "#1baf7a",  # aqua
    "CEDIDAS_IAXIS":     "#eda100",  # amarillo
    "CEDIDAS_TERREMOTO": "#008300",  # verde
}

# Rampa secuencial (un solo tono, claro → oscuro) para el mapa de calor
RAMPA_AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# Colores de estado (reservados; siempre van con ícono + texto, nunca solo color)
STATUS_GOOD, STATUS_WARNING, STATUS_CRITICAL = "#0ca30c", "#fab219", "#d03b3b"


def layout_base(fig, titulo, alto=420):
    fig.update_layout(
        title=dict(text=titulo, font=dict(color=INK, size=16)),
        font=dict(family=FONT, color=INK_2, size=12),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        height=alto, margin=dict(l=10, r=10, t=50, b=10),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=MUTED), zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=MUTED), zeroline=False)
    return fig

### 5.1 Resumen del último mes (KPIs)

La foto rápida del periodo más reciente: ¿puedo correr el cierre tranquilo o hay que avisar a alguien?
Se agregan dos tarjetas nuevas: cuántas variaciones atípicas de valor hay, y cuántos huecos de carga.

In [20]:
ultimo_con_datos = int(df["periodo_contable_analisis"].max())
d_ult = df[df["periodo_contable_analisis"] == ultimo_con_datos]

pct_global = 100.0 * d_ult["con_valor"].sum() / d_ult["total_registros"].sum()
n_alerta   = int((d_ult["semaforo"] == "ALERTA").sum())
n_completo = int((d_ult["semaforo"] == "COMPLETO").sum())

if n_alerta > 0:
    peor = (d_ult[d_ult["semaforo"] == "ALERTA"].groupby("fuente")["sin_valor"].sum().idxmax())
    estado_icono, estado_color, estado_texto = "⚠", STATUS_CRITICAL, f"{n_alerta} en ALERTA"
else:
    peor = "—"
    estado_icono, estado_color, estado_texto = "✓", STATUS_GOOD, "Sin alertas"

def tile(etiqueta, valor, sub=""):
    return f"""
    <div style="flex:1;min-width:170px;background:{SURFACE};border:1px solid rgba(11,11,11,0.10);
                border-radius:8px;padding:14px 16px;font-family:{FONT};">
      <div style="font-size:12px;color:{MUTED};margin-bottom:4px;">{etiqueta}</div>
      <div style="font-size:26px;color:{INK};font-weight:600;">{valor}</div>
      <div style="font-size:12px;color:{INK_2};margin-top:2px;">{sub}</div>
    </div>"""

html_kpis = f"""
<div style="display:flex;gap:12px;flex-wrap:wrap;">
  {tile("Último periodo con datos", ultimo_con_datos)}
  {tile("Completitud global del mes", f"{pct_global:.2f}%", "con_valor / total_registros")}
  {tile("Estado del semáforo",
        f'<span style="color:{estado_color};">{estado_icono}</span> {estado_texto}',
        f"{n_completo} combinaciones COMPLETO")}
  {tile("Fuente más afectada (completitud)", peor, "por registros sin valor" if peor != "—" else "")}
  {tile("Variaciones atípicas de valor", len(alertas_valor), f"de {len(variacion_valor)} combinaciones revisadas")}
  {tile("Huecos de carga", len(huecos), "combinación sin filas en algún mes")}
</div>"""

display(HTML(html_kpis))

### 5.2 Completitud — mapa de calor (12 meses)

Cada fila es una combinación fuente + cuenta + libro; cada columna es un mes. El color muestra el
**% de registros SIN valor**. Celda vacía (hueco) = no llegó ninguna fila ese mes.

In [21]:
piv_sin  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="pct_sin_valor", aggfunc="max")
piv_tot  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
piv_nul  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="sin_valor", aggfunc="sum")

piv_sin = piv_sin.reindex(columns=PERIODOS)
piv_tot = piv_tot.reindex(columns=PERIODOS)
piv_nul = piv_nul.reindex(columns=PERIODOS)
piv_sin = piv_sin.sort_index(ascending=False)
piv_tot = piv_tot.reindex(piv_sin.index)
piv_nul = piv_nul.reindex(piv_sin.index)

zmax = max(float(piv_sin.max().max() or 0), 5.0)  # escala mínima 5% para que no explote con valores diminutos

hover = []
for s in piv_sin.index:
    fila = []
    for p in PERIODOS:
        v, t, n = piv_sin.loc[s, p], piv_tot.loc[s, p], piv_nul.loc[s, p]
        if pd.isna(v):
            fila.append(f"<b>{s}</b><br>{p}<br>⚠ Sin filas cargadas este mes")
        else:
            fila.append(f"<b>{s}</b><br>{p}<br>Registros: {int(t):,}<br>"
                        f"Sin valor: {int(n):,} ({v:.2f}%)<br>Completitud: {100 - v:.2f}%")
    hover.append(fila)

escala = [[i / (len(RAMPA_AZUL) - 1), c] for i, c in enumerate(RAMPA_AZUL)]

fig_heatmap = go.Figure(go.Heatmap(
    z=piv_sin.values,
    x=[str(p) for p in PERIODOS],
    y=list(piv_sin.index),
    colorscale=escala, zmin=0, zmax=zmax,
    xgap=2, ygap=2,  # separación de 2px entre celdas
    hoverinfo="text", text=hover,
    colorbar=dict(title=dict(text="% sin valor", font=dict(color=INK_2)),
                  tickfont=dict(color=MUTED), outlinewidth=0),
))
layout_base(fig_heatmap, "% de registros SIN valor por cuenta/libro y mes (más oscuro = peor)",
            alto=max(380, 32 * len(piv_sin) + 120))
fig_heatmap.show()

### 5.3 Completitud — tendencia por fuente

Porcentaje de completitud de cada fuente mes a mes. Lo esperado es una línea plana pegada al 100%;
cualquier caída es un mes con problemas.

In [22]:
tend = (df.groupby(["fuente", "periodo_contable_analisis"])
          .agg(con=("con_valor", "sum"), tot=("total_registros", "sum"))
          .reset_index())
tend["pct"] = 100.0 * tend["con"] / tend["tot"]

fig_trend = go.Figure()
for fuente, color in COLOR_FUENTE.items():
    d = tend[tend["fuente"] == fuente].sort_values("periodo_contable_analisis")
    if d.empty:
        continue
    fig_trend.add_trace(go.Scatter(
        x=d["periodo_contable_analisis"].astype(str), y=d["pct"],
        name=fuente, mode="lines+markers",
        line=dict(color=color, width=2), marker=dict(size=8, color=color),
        hovertemplate="<b>%{fullData.name}</b><br>%{x}<br>Completitud: %{y:.2f}%<extra></extra>",
    ))
    fig_trend.add_annotation(x=str(d["periodo_contable_analisis"].iloc[-1]), y=d["pct"].iloc[-1],
                       text=fuente, font=dict(color=INK_2, size=11),
                       xanchor="left", xshift=8, showarrow=False)

layout_base(fig_trend, "Completitud (%) por fuente — últimos 12 meses", alto=420)
fig_trend.update_yaxes(ticksuffix="%")
fig_trend.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, font=dict(color=INK_2)),
                  margin=dict(r=140))
fig_trend.show()

### 5.4 Variación — valor total con anomalías marcadas

La línea muestra el valor total de la reserva contable por fuente. El ⚠ marca los meses donde,
dentro de esa fuente, alguna combinación cuenta/libro tuvo una variación atípica (ver la definición en
la sección 4). El detalle completo — cuál cuenta, cuál libro, cuánto cambió — está en la tabla de
"Alertas de variación — valor" más abajo.

In [23]:
tend_valor = (df_valor.groupby(["fuente", "periodo_contable_analisis"])["valor_total"]
              .sum().reset_index())
piv_valor_fuente = (tend_valor.pivot_table(index="fuente", columns="periodo_contable_analisis",
                                            values="valor_total", aggfunc="sum")
                    .reindex(columns=PERIODOS))
variacion_fuente = marcar_variacion_atipica(piv_valor_fuente, PERIODOS, nombre_id="fuente")

fig_valor = go.Figure()
for fuente, color in COLOR_FUENTE.items():
    d = tend_valor[tend_valor["fuente"] == fuente].sort_values("periodo_contable_analisis")
    if d.empty:
        continue
    fig_valor.add_trace(go.Scatter(
        x=d["periodo_contable_analisis"].astype(str), y=d["valor_total"],
        name=fuente, mode="lines+markers",
        line=dict(color=color, width=2), marker=dict(size=8, color=color),
        hovertemplate="<b>%{fullData.name}</b><br>%{x}<br>Valor total: %{y:,.0f}<extra></extra>",
    ))
    fig_valor.add_annotation(x=str(d["periodo_contable_analisis"].iloc[-1]), y=d["valor_total"].iloc[-1],
                              text=fuente, font=dict(color=INK_2, size=11),
                              xanchor="left", xshift=8, showarrow=False)

atipicos_fuente = variacion_fuente[variacion_fuente["variacion_atipica"]]
if not atipicos_fuente.empty:
    fig_valor.add_trace(go.Scatter(
        x=atipicos_fuente["periodo_contable_analisis"].astype(str),
        y=atipicos_fuente["valor"],
        mode="markers", name="⚠ Variación atípica",
        marker=dict(size=14, color=STATUS_WARNING, symbol="diamond",
                    line=dict(width=2, color=INK)),
        hovertemplate="<b>⚠ Variación atípica</b><br>%{x}<br>Valor: %{y:,.0f}<extra></extra>",
    ))

layout_base(fig_valor, "Valor total de la reserva contable por fuente — últimos 12 meses", alto=420)
fig_valor.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, font=dict(color=INK_2)),
                        margin=dict(r=140))
fig_valor.show()

### 5.5 Tablas de alertas y huecos

El detalle accionable: cada fila es un aviso pendiente — a quién avisar y por qué.

In [24]:
alertas = (df[df["semaforo"] == "ALERTA"]
           [["fuente", "cuenta", "libro", "periodo_contable_analisis",
             "total_registros", "con_valor", "sin_valor", "pct_completitud"]]
           .sort_values(["sin_valor", "periodo_contable_analisis"], ascending=[False, False])
           .reset_index(drop=True))


def _fmt_num(v):
    return "" if pd.isna(v) else f"{v:,.0f}"


def _fmt_dec(v):
    return "" if pd.isna(v) else f"{v:,.2f}"


def _fmt_pct(v):
    if pd.isna(v):
        return ""
    if v in (float("inf"), float("-inf")):
        return "∞" if v > 0 else "-∞"
    return f"{v:+.1%}"


def formatear_tabla(df_in, enteros=(), decimales=(), porcentajes=()):
    d = df_in.copy()
    for c in enteros:
        d[c] = d[c].map(_fmt_num)
    for c in decimales:
        d[c] = d[c].map(_fmt_dec)
    for c in porcentajes:
        d[c] = d[c].map(_fmt_pct)
    return d


def tabla_html(df_in, mensaje_vacio="Sin registros — nada que revisar."):
    if df_in.empty:
        return f'<p style="color:{STATUS_GOOD};font-family:{FONT};">✓ {mensaje_vacio}</p>'
    return df_in.to_html(index=False, border=0, classes="tabla")


tabla_completitud = formatear_tabla(
    alertas[["fuente", "cuenta", "libro", "periodo_contable_analisis",
             "total_registros", "con_valor", "sin_valor", "pct_completitud"]],
    enteros=["total_registros", "con_valor", "sin_valor"],
    decimales=["pct_completitud"],
)

tabla_valor = formatear_tabla(
    alertas_valor[["serie", "periodo_contable_analisis", "valor", "z_score", "pct_variacion"]]
        .rename(columns={"serie": "fuente · cuenta · libro"}),
    decimales=["valor", "z_score"], porcentajes=["pct_variacion"],
)

tabla_volumen = formatear_tabla(
    alertas_volumen[["serie", "periodo_contable_analisis", "valor", "z_score", "pct_variacion"]]
        .rename(columns={"serie": "fuente · cuenta · libro", "valor": "total_registros"}),
    enteros=["total_registros"], decimales=["z_score"], porcentajes=["pct_variacion"],
)

tabla_huecos = formatear_tabla(
    huecos.rename(columns={"serie": "fuente · cuenta · libro"})
)

for titulo, tabla in [
    ("Alertas de completitud", tabla_completitud),
    ("Alertas de variación — valor", tabla_valor),
    ("Alertas de variación — volumen", tabla_volumen),
    ("Huecos de carga", tabla_huecos),
]:
    display(HTML(f"<h4>{titulo}</h4>" + tabla_html(tabla)))

fuente · cuenta · libro,periodo_contable_analisis,valor,z_score,pct_variacion
CEDIDAS_AS400 · 510305 · AA,202606,"808,395,380.00",1.41,+357.5%
CEDIDAS_IAXIS · 510305 · AL,202606,"444,160,815.00",0.37,-66.6%
DIRECTA_IAXIS · 510310 · AA,202606,"2,259,010,049.00",1.37,+59.3%
CEDIDAS_TERREMOTO · 510305 · AL,202606,"-595,856,168.00",0.19,+86.9%
CEDIDAS_TERREMOTO · 510305 · AA,202606,"-489,667,985.00",-1.12,-314.2%
CEDIDAS_TERREMOTO · 410305 · AL,202606,"1,135,411,702.00",0.24,-29.1%
CEDIDAS_TERREMOTO · 410305 · AA,202606,"-5,096,016.00",0.04,-100.2%
CEDIDAS_TERREMOTO · 263005 · AL,202606,"-539,555,534.00",-0.51,-118.4%
CEDIDAS_TERREMOTO · 263005 · AA,202606,"494,764,001.00",0.29,+117.9%
DIRECTA_IAXIS · 410310 · AA,202606,"-762,484,996.00",0.18,+23.1%


fuente · cuenta · libro,periodo_contable_analisis,total_registros,z_score,pct_variacion
CEDIDAS_TERREMOTO · 263005 · AL,202606,"90,568",-0.48,-25.9%
CEDIDAS_TERREMOTO · 263005 · AA,202606,32,-0.69,-96.0%
CEDIDAS_TERREMOTO · 410305 · AA,202606,1,-0.65,-99.9%
CEDIDAS_TERREMOTO · 510305 · AL,202606,"26,503",-0.19,+21.0%
CEDIDAS_TERREMOTO · 510305 · AA,202606,31,-0.56,-68.7%
CEDIDAS_TERREMOTO · 410305 · AL,202606,"64,065",-0.35,-36.1%
DIRECTA_IAXIS · 410310 · AA,202606,"9,570",-0.23,-25.1%
CEDIDAS_TERREMOTO · 510305 · AL,202605,"21,898",-0.30,+87.1%
DIRECTA_IAXIS · 510315 · AL,202605,"29,679",-0.62,+21.3%
CEDIDAS_TERREMOTO · 510305 · AA,202604,90,-0.55,+143.2%


### 5.6 Ensamblar y guardar el archivo HTML final

Juntamos todo lo anterior (KPIs, gráficos, tablas) en una sola página HTML con estilos propios, y la
guardamos junto a este notebook.

In [25]:
import plotly.io as pio
from pathlib import Path


def fig_html(fig, incluir_js):
    return pio.to_html(fig, full_html=False, include_plotlyjs=incluir_js,
                        config={"displaylogo": False, "responsive": True})


fecha_generacion = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

html_doc = f"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Tablero de completitud y variación — DWH</title>
<style>
  :root {{
    --surface-1: {SURFACE}; --page: #f9f9f7; --ink-1: {INK}; --ink-2: {INK_2};
    --muted: {MUTED}; --grid: {GRID}; --border: rgba(11,11,11,0.10);
  }}
  @media (prefers-color-scheme: dark) {{
    :root {{
      --surface-1: #1a1a19; --page: #0d0d0d; --ink-1: #ffffff; --ink-2: #c3c2b7;
      --muted: #898781; --grid: #2c2c2a; --border: rgba(255,255,255,0.10);
    }}
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; padding: 24px; background: var(--page); color: var(--ink-1);
    font-family: {FONT};
  }}
  h1 {{ font-size: 22px; margin: 0 0 4px 0; }}
  h2 {{ font-size: 17px; margin: 0 0 12px 0; }}
  p.subtitulo {{ color: var(--ink-2); margin: 0 0 24px 0; font-size: 13px; }}
  section {{
    background: var(--surface-1); border: 1px solid var(--border); border-radius: 10px;
    padding: 16px 20px; margin-bottom: 20px;
  }}
  p.ok {{ color: {STATUS_GOOD}; }}
  table.tabla {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
  table.tabla th, table.tabla td {{
    border-bottom: 1px solid var(--grid); padding: 6px 10px; text-align: left;
    color: var(--ink-2);
  }}
  table.tabla th {{ color: var(--muted); font-weight: 600; }}
  .nota {{ font-size: 12px; color: var(--muted); margin-top: 8px; }}
  .tabla-guia td, .tabla-guia th {{ vertical-align: top; }}
</style>
</head>
<body>
  <h1>Tablero de completitud y variación — DWH</h1>
  <p class="subtitulo">
    Ventana analizada: {PERIODOS[0]} → {PERIODOS[-1]} (12 meses) · Generado: {fecha_generacion}
  </p>

  <section>
    <h2>Resumen del último mes</h2>
    {html_kpis}
  </section>

  <section>
    <h2>Completitud — mapa de calor (12 meses)</h2>
    {fig_html(fig_heatmap, True)}
  </section>

  <section>
    <h2>Completitud — tendencia por fuente</h2>
    {fig_html(fig_trend, False)}
  </section>

  <section>
    <h2>Variación — valor total de la reserva contable por fuente</h2>
    {fig_html(fig_valor, False)}
    <p class="nota">
      ⚠ marca meses donde alguna combinación fuente + cuenta + libro se movió más de lo esperado
      (más de {UMBRAL_Z_VARIACION:g} desviaciones estándar respecto a su propio promedio en la
      ventana, o más de {UMBRAL_PCT_VARIACION:.0%} respecto al mes anterior). Umbrales de primera
      versión — pendientes de validar con el equipo (ver detalle en la tabla de abajo).
    </p>
  </section>

  <section>
    <h2>Alertas de completitud (registros con valor vacío)</h2>
    {tabla_html(tabla_completitud)}
  </section>

  <section>
    <h2>Alertas de variación — valor</h2>
    {tabla_html(tabla_valor)}
  </section>

  <section>
    <h2>Alertas de variación — volumen (cantidad de registros)</h2>
    {tabla_html(tabla_volumen)}
  </section>

  <section>
    <h2>Huecos de carga (combinación sin ninguna fila en algún mes del rango)</h2>
    {tabla_html(tabla_huecos)}
  </section>

  <section>
    <h2>Cómo interpretar este tablero</h2>
    <table class="tabla tabla-guia">
      <tr><th>Si ves…</th><th>Significa…</th><th>Qué hacer</th></tr>
      <tr><td>KPIs en verde, heatmap claro</td><td>Todo completo</td><td>OK para el cierre del día 10</td></tr>
      <tr><td>Celda oscura en el heatmap</td><td>Esa cuenta/libro tiene registros sin valor ese mes</td><td>Avisar al dueño de la tabla</td></tr>
      <tr><td>Celda vacía (hueco) en el heatmap</td><td>No llegó ninguna fila ese mes</td><td>Revisar si la carga de esa fuente corrió</td></tr>
      <tr><td>⚠ en el gráfico de variación</td><td>El valor total de esa fuente se movió más de lo usual</td><td>Revisar la tabla de alertas de variación — puede ser normal (crecimiento real) o un error de carga</td></tr>
      <tr><td>Fila en "huecos de carga"</td><td>Esa combinación específica no tuvo ningún registro ese mes</td><td>Confirmar si esa fuente debía o no tener datos ese mes</td></tr>
    </table>
    <p class="nota">
      Este tablero valida existencia (completitud) y variación frente al propio histórico — no valida
      si el número es "correcto" en un sentido contable. Cuentas excluidas por diseño: Incurrido,
      Salvamentos, Recobros, IVA AG (+ 2 cuentas pendientes de confirmar con Andrey).
    </p>
  </section>
</body>
</html>
"""

ruta_html = Path("dashboard_completitud.html").resolve()
ruta_html.write_text(html_doc, encoding="utf-8")
print(f"Tablero guardado en: {ruta_html}")

Tablero guardado en: C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\data-completeness-validator\notebooks\dashboard_completitud.html


## 6. Dónde quedó el resultado

El archivo `dashboard_completitud.html` (guardado junto a este notebook) trae todo lo de arriba —
KPIs, mapa de calor, tendencias, variación con anomalías marcadas, y las 4 tablas de alertas — en un
solo archivo que **no necesita Python ni conexión al DWH** para verse: ábrelo con doble clic en
cualquier navegador, o adjúntalo a un correo/ticket.

Recordatorios:
- Este control solo valida que el dato **exista** (completitud) y que no se haya movido más de lo
  usual (variación) — no valida si el número es "correcto" en un sentido contable.
- Cuentas excluidas por diseño: Incurrido, Salvamentos, Recobros, IVA AG — **+ 2 cuentas pendientes de
  confirmar con Andrey**.
- Los umbrales de variación (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION`) son una primera propuesta —
  todavía hay que validarlos con el equipo (es la "Fase 2" mencionada en
  `Hallazgos y Estado del Proyecto.md`).